# AI-Powered Business Intelligence & Data Analytics Project
**Author:** Pratiksha Ingale

This notebook contains the complete pipeline: Data Cleaning -> EDA -> KPIs -> Trend Analysis -> Forecasting -> Insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## 1. Data Cleaning & Understanding

In [ ]:
# Load raw dataset
url = 'https://raw.githubusercontent.com/pplonski/datasets-for-start/refs/heads/master/superstore-sales/superstore_dataset2011-2015.csv'
df_raw = pd.read_csv(url, encoding='latin1')

print(f"Raw Shape: {df_raw.shape}")

# Clean column names
df_raw.columns = df_raw.columns.str.strip().str.replace(' ', '_').str.replace('-', '_').str.lower()

# Convert dates
df_raw['order_date'] = pd.to_datetime(df_raw['order_date'], format='mixed', errors='coerce')

# Clean numerical values (remove currency symbols if any)
for col in ['sales', 'profit', 'discount', 'shipping_cost']:
    if df_raw[col].dtype == 'object':
        df_raw[col] = df_raw[col].astype(str).str.replace(r'[^\d.-]', '', regex=True)
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    df_raw[col] = df_raw[col].fillna(0)

# Drop rows with invalid dates
df = df_raw.dropna(subset=['order_date']).copy()
print(f"Cleaned Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['sales'], bins=50, kde=True)
plt.title('Distribution of Sales')
plt.xlim(0, 2000) # Zoomed in for better visibility
plt.show()

In [ ]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
plt.figure(figsize=(10,8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## 3. KPI Engine & Driver Analysis

In [ ]:
kpis = {
    'Total Revenue': df['sales'].sum(),
    'Total Profit': df['profit'].sum(),
    'Total Orders': df['order_id'].nunique(),
    'Profit Margin (%)': (df['profit'].sum() / df['sales'].sum() * 100)
}

print("--- Key Performance Indicators ---")
for k, v in kpis.items():
    print(f"{k}: {v:,.2f}")

print("\n--- Top 3 Profitable Categories ---")
print(df.groupby('category')['profit'].sum().sort_values(ascending=False).head(3))

## 4. Forecasting Model

In [ ]:
# Prepare monthly sales data
monthly_sales = df.groupby('order_date')['sales'].sum().reset_index()
monthly_sales = monthly_sales.set_index('order_date').resample('ME').sum().reset_index()

monthly_sales['year'] = monthly_sales['order_date'].dt.year
monthly_sales['month'] = monthly_sales['order_date'].dt.month
monthly_sales['time_idx'] = np.arange(len(monthly_sales))

X = monthly_sales[['time_idx', 'month']]
y = monthly_sales['sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R2 Score: {r2_score(y_test, y_pred):.2f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")

# Forecast next 6 months
last_date = monthly_sales['order_date'].max()
last_idx = monthly_sales['time_idx'].max()

future_dates = [last_date + pd.DateOffset(months=i) for i in range(1, 7)]
future_data = pd.DataFrame({
    'order_date': future_dates,
    'year': [d.year for d in future_dates],
    'month': [d.month for d in future_dates],
    'time_idx': [last_idx + i for i in range(1, 7)]
})
future_data['predicted_sales'] = model.predict(future_data[['time_idx', 'month']])
print("\n--- 6-Month Forecast ---")
print(future_data[['order_date', 'predicted_sales']])

## 5. AI Insight Generation

In [ ]:
def generate_insights(df, kpis):
    margin = kpis['Profit Margin (%)']
    print("\n[AI INSIGHT 1]")
    print(f"OBSERVATION: Total revenue is ${kpis['Total Revenue']:,.2f} with a margin of {margin:.2f}%.")
    print("RECOMMENDATION:", "Maintain strategy" if margin > 10 else "Optimize costs immediately.")
    
    # Risk detection
    sub_cat = df.groupby('sub_category').agg({'sales':'sum', 'profit':'sum'}).reset_index()
    loss_makers = sub_cat[sub_cat['profit'] < 0]
    if not loss_makers.empty:
        worst = loss_makers.sort_values('profit').iloc[0]
        print("\n[AI INSIGHT 2]")
        print(f"RISK OBSERVATION: Sub-Category '{worst['sub_category']}' is operating at a loss (${worst['profit']:,.2f}).")
        print("RECOMMENDATION: Investigate discount strategies and potential discontinuance.")

generate_insights(df, kpis)